# Preparación de datos demográficos de OWID (2000–2022)
# Montserrat Hernández Zarate


**Objetivo:** descargar cuatro conjuntos de datos, revisar sus columnas, preparar cada uno por separado y unirlos por `Entity` y `Year`. El resultado servirá para una práctica posterior de aprendizaje automático. (variable objetivo)

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import requests

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"

## 1. Descargar los cuatro archivos

Las direcciones son cadenas de texto simples. Cada archivo se carga en su propio DataFrame.

In [0]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO

population = pd.read_csv(url_population, storage_options=OPCIONES_OWID)
births = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
deaths = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
child_deaths = pd.read_csv(url_child_deaths, storage_options=OPCIONES_OWID)

print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)

## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales antes de seleccionarlas.

In [0]:
for nombre, tabla in [("population", population), 
                      ("births", births),
                      ("deaths", deaths), 
                      ("child_deaths", child_deaths)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [0]:
columnas_population = [
    columna for columna in population #extraer las columnas que solamente quiera yo
        if "population" in columna and "estimates" in columna] #pq las tiene el mi variable estimada

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba" #cacha un error (no se encontro la columna)


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [0]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [0]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [0]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]] 

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

## 7. Revisar llaves y cobertura

Cada combinación `Entity`–`Year` debe aparecer una sola vez en cada tabla. `Code` no se utiliza como llave porque puede faltar en agregados regionales.

In [0]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set(population_limpio["entity"]) & set(births_limpio["entity"]) \
                  & set(deaths_limpio["entity"]) & set(child_deaths_limpio["entity"])

print("Entidades presentes en los cuatro archivos:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 8. Unir las tablas, un paso a la vez

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas.

In [0]:
population_births = pd.merge( #se integran cada una de las variables proyectoras en una sola data center
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [0]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [0]:
demografia2222 = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", demografia.shape)
display(demografia.head())

In [0]:
variable_gdp = pd.merge(
    demografia, df_capita_limpio,  
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", variable_gdp.shape)    
display(variable_gdp.head())

In [0]:
variable_health = pd.merge(
    variable_gdp, df_healthcare_limpio,  
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", variable_health.shape)    
display(variable_health.head())

In [0]:
variable_life = pd.merge(
    variable_gdp, df_life_limpio,  
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", variable_life.shape)    
display(variable_life.head())

## 9. Revisar faltantes y crear tasas

Eliminamos solo filas con algún indicador necesario ausente. Las tasas de nacimientos y defunciones se expresan por 1,000 habitantes. La razón de muertes de menores de cinco años usa nacimientos del **mismo año** como denominador; es una medida descriptiva aproximada, no una tasa de mortalidad de cohorte.

In [0]:
print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia2 = demografia.dropna(subset=["population", "births", "deaths", "child_deaths_under_5"]).copy()
demografia2 = demografia[(demografia2["population"] > 0) & (demografia2["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia2))
print("Entidades después de limpiar:", demografia2["entity"].nunique())
assert not demografia2.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [0]:
demografia2["birth_rate_per_1000"] = demografia2["births"] / demografia2["population"] * 1000
demografia2["death_rate_per_1000"] = demografia2["deaths"] / demografia2["population"] * 1000
demografia2["under5_deaths_per_1000_births_same_year"] = (
    demografia2["child_deaths_under_5"] / demografia2["births"] * 1000
)

demografia2 = demografia2[[
    "entity", "year", "population", "births", "deaths", "child_deaths_under_5",
    "birth_rate_per_1000", "death_rate_per_1000",
    "under5_deaths_per_1000_births_same_year"
]].sort_values(["entity", "year"]).reset_index(drop=True)

display(demografia2.head())
display(demografia2.describe(include="all"))

## 10. Visualizar una entidad disponible

Puedes cambiar `entidad` por cualquier nombre de `demografia["Entity"].unique()`.

In [0]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year")

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 11. Guardar el resultado

El CSV se guarda junto al notebook. Contiene solamente entidades y años compartidos por las cuatro fuentes.

In [0]:
from pathlib import Path

carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "demografia_owid_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)}")

In [0]:
#GDP per capita 2025
#GDP per capita 2025. GDP per capita is a country's gross domestic product divided by its population. This data is adjusted for inflation and differencesin living costs between countries.
  
# Fetch the data.

df_capita = pd.read_csv("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true%22", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
 
# Fetch the metadata

metadata = requests.get("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.metadata.json?v=1&csvType=full&useColumnShortNames=true%22).json()")

df_capita.head(5)

In [0]:
df_capita.columns = df_capita.columns.str.lower()

columnas_capita = [c for c in df_capita.columns
                  if c not in ["entity", "code", "year", "owd"]] 

print("Columnas candidatas:", columnas_capita)


col_capita = columnas_capita[0]

df_capita_limpio = df_capita[["entity", "year", col_capita]].copy()
df_capita_limpio = df_capita_limpio.rename(columns={col_capita: "gdp"})

df_capita_limpio = df_capita_limpio[df_capita_limpio["year"].between(2000, 2022)].copy()

print(df_capita_limpio.shape)
display(df_capita_limpio.head())


In [0]:
#Total healthcare spending as a share of GDP
#Total healthcare spending as a share of GDP 2000 to 2023. Total healthcare expenditure as the share of national gross domestic product (GDP).

#Fetch the data.
df_healthcare = pd.read_csv("https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true%22", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

#Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.metadata.json?v=1&csvType=full&useColumnShortNames=true%22).json()")
 
df_healthcare.head(5)

In [0]:
df_healthcare.columns = df_healthcare.columns.str.lower()

columnas_health = [c for c in df_healthcare.columns
                  if c not in ["entity", "code", "year"]]
 
print("Columnas candidatas:", columnas_health)
  
col_health = columnas_health[0]
 
df_healthcare_limpio = df_healthcare[["entity", "year", col_health]].copy()
df_healthcare_limpio = df_healthcare_limpio.rename(columns={col_health: "percentage_health"})
 
df_healthcare_limpio = df_healthcare_limpio[df_healthcare_limpio["year"].between(2000, 2022)].copy()
 
print(df_healthcare_limpio.shape)
display(df_healthcare_limpio.head())


In [0]:
df_life = pd.read_csv("https://ourworldindata.org/grapher/life-expectancy.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

df_life.head(5)

In [0]:
df_life.columns = df_life.columns.str.lower()

columnas_health = [c for c in df_life.columns
                  if c not in ["entity", "code", "year"]]
 
print("Columnas candidatas:", columnas_health)
  
col_life = columnas_health[0]
 
df_life_limpio = df_life[["entity", "year", col_life]].copy()
df_life_limpio = df_life_limpio.rename(columns={col_life: "life_expectancy"})
 
df_life_limpio = df_life_limpio[df_life_limpio["year"].between(2000, 2022)].copy()
 
print(df_life_limpio.shape)
display(df_life_limpio.head())

In [0]:
from pathlib import Path

carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "demografia_owid_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)}")

In [0]:
#Para guardar el dataset en data/raw
# Guardar el df completo
 
from pathlib import Path
 
carpeta_salida = Path("../data/raw")
carpeta_salida.mkdir(parents=True, exist_ok=True)
 
salida = carpeta_salida / "dataset_7variables_2000_2022_V2.csv"
variable_life.to_csv(salida, index=False)
 
print(f"Archivo guardado: {salida.resolve()} \n | filas: {len(variable_life)}")
 